# **Evaluation Pipeline**

#### Note: For all types of score generations, the parameter "bio" must match the world leader's name/title exactly as in the [multi-fact paper](https://arxiv.org/pdf/2402.18045) (page 17)


### **Initial set-up**

In [ ]:
# Version comptability with Multi-fact repo
!pip install transformers==4.41.0 sentence-transformers==3.0.0 -q

In [ ]:
import os
import sys

os.chdir("/content/multilingual-hallucination")  # path to multilingual-hallucination git
sys.path.append(os.getcwd())  # add correct path
print(os.getcwd())  # sanity check

In [ ]:
import json, os

os.environ["CUDA_VISIBLE_DEVICES"] = "0" # set you GPU here, optional

In [ ]:
from factscore.factscorer import FactScorer

# Initialize factscorer
# Refer to README.md for instructions on installing the wikipedia database
fs = FactScorer(openai_key="./openai-api.txt", data_dir="/path/to/wikipedia/database",
                fact_generation_type="Mistral", model_name="retrieval+mistral"
                )

# You have the folliowing options for fact_generation_type: InstructGPT (make sure you have a valid openai_key), or Mistral
# You have the following options for model_name: "retrieval+ChatGPT", "npm", "retrieval+ChatGPT+npm", "retrieval+mistral", "retrieval+mistral+npm"
# in the data_dir, put the path where you put your wikipedia database

## **Baseline Scores**

In [ ]:
def print_factscore_baseline(lang : str, bio : str):
  file = f"generated_bios_qwen/translated2en_8B/{lang}_2en_bios.json"
  with open(file, "r") as f:
    bios = json.load(f)

  topic = bio
  bio = bios[topic]

  out = fs.get_score([topic], [bio], gamma=20)
  print(f"Language: {lang}\nTopic: {topic}\nFActScore: {out['score']}\nFActScore w/o length penalty: {out['init_score']}\n% of responding (not abstaining from answering): {out['respond_ratio']}\nnum of facts per response: {out['num_facts_per_response']}\n")

In [ ]:
print_factscore_baseline("English", "Xi Jinping")


In [ ]:
print_factscore_baseline("Hindi", "Xi Jinping")

In [ ]:
print_factscore_baseline("Bengali", "Xi Jinping")

In [ ]:
print_factscore_baseline("Swahili", "Xi Jinping")

## **RAG Scores**

In [ ]:
def print_rag_factscore(lang : str, bio : str, crosslingual : bool):
  if crosslingual:
    file = f"generated_bios_qwen_rag/crosslingual_rag/translated2en/{lang}_2en_bios.json"
  else:
    file = f"generated_bios_qwen_rag/monolingual_rag/translated2en/{lang}_2en_bios.json"

  with open(file, "r") as f:
    bios = json.load(f)

  topic = bio
  bio = bios[topic]

  rag_mode = ""
  if crosslingual:
    rag_mode = "w/ English Wiki"
  else:
    rag_mode = "w/ Native Wiki"

  out = fs.get_score([topic], [bio], gamma=20)
  print(f"Language: {lang} {rag_mode}\nTopic: {topic}\nFActScore: {out['score']}\nFActScore w/o length penalty: {out['init_score']}\n% of responding (not abstaining from answering): {out['respond_ratio']}\nnum of facts per response: {out['num_facts_per_response']}\n")

#### Crosslingual RAG Scores

In [ ]:
print_rag_factscore("English", "Xi Jinping", True)

In [ ]:
print_rag_factscore("Hindi", "Xi Jinping", True)

In [ ]:
print_rag_factscore("Bengali", "Xi Jinping", True)

In [ ]:
print_rag_factscore("Swahili", "Xi Jinping", True)

#### Monolingual RAG Scores

In [ ]:
print_rag_factscore("English", "Xi Jinping", False)

In [ ]:
print_rag_factscore("Hindi", "Xi Jinping", False)

In [ ]:
print_rag_factscore("Bengali", "Xi Jinping", False)

In [ ]:
print_rag_factscore("Swahili", "Xi Jinping", False)

## **CoT Scores**

In [ ]:
def print_factscore_cot(lang : str, bio : str, version : str):
  file = f"generated_bios_qwen_CoT/{version}/translated2en/{lang}_2en_bios.json"
  with open(file, "r") as f:
    bios = json.load(f)

  topic = bio
  bio = bios[topic]

  out = fs.get_score([topic], [bio], gamma=20)
  print(f"Language: {lang}\nTopic: {topic}\nFActScore: {out['score']}\nFActScore w/o length penalty: {out['init_score']}\n% of responding (not abstaining from answering): {out['respond_ratio']}\nnum of facts per response: {out['num_facts_per_response']}\n")

#### Version 1 (Vanilla Zero-shot Prompting) Scores

To run scores on bios generated with vanilla zero-shot prompts, call print_factscore_cot() function with version = "version1"

In [ ]:
print_factscore_cot("English", "Xi Jinping", "version1")

In [ ]:
print_factscore_cot("Hindi", "Xi Jinping", "version1")

In [ ]:
print_factscore_cot("Bengali", "Xi Jinping", "version1")

In [ ]:
print_factscore_cot("Swahili", "Xi Jinping", "version1")

#### Version 2 (Plan and Structure Zero-Shot Prompting) Scores

To run scores on bios generated with vanilla zero-shot prompts, call print_factscore_cot() function with version = "version2"

In [ ]:
print_factscore_cot("English", "Xi Jinping", "version2")

In [ ]:
print_factscore_cot("Hindi", "Xi Jinping", "version2")

In [ ]:
print_factscore_cot("Bengali", "Xi Jinping", "version2")

In [ ]:
print_factscore_cot("Swahili", "Xi Jinping", "version2")

## **Crosslingual RAG with GPT pre-translated context: Error Analysis**

In [ ]:
def print_factscore_context_pretranslated(lang : str, bio : str):
  file = f"generated_bios_qwen_rag/crosslingual_rag/context_pre-translated/translated2en/{lang}_2en_bios.json"
  with open(file, "r") as f:
    bios = json.load(f)

  topic = bio
  bio = bios[topic]

  out = fs.get_score([topic], [bio], gamma=20)
  print(f"Language: {lang}\nTopic: {topic}\nFActScore: {out['score']}\nFActScore w/o length penalty: {out['init_score']}\n% of responding (not abstaining from answering): {out['respond_ratio']}\nnum of facts per response: {out['num_facts_per_response']}\n")

In [ ]:
print_factscore_context_pretranslated("Hindi", "Xi Jinping")

In [ ]:
print_factscore_context_pretranslated("Bengali", "Xi Jinping")

In [ ]:
print_factscore_context_pretranslated("Swahili", "Xi Jinping")